## Module 2: Vector Search

This module extends the RAG pipeline from the first module with vector search. Vector search matches documents by semantic meaning instead of exact keyword overlap. Traditional keyword search matches exact words, whereas vector search utilizes a neural embedding model to convert text into numerical arrays (vectors) that capture the actual semantic meaning of a query. This process operates in two stages: 

1. Offline indexing phase that stores document vectors
2. Online querying phase that translates a user's question to find the closest matches based on a distance metric like cosine similarity 

While vector search is superior for natural language and paraphrasing, it introduces significant operational complexity and cost; therefore, the best approach is to start with text search and eventually combine both methods into a hybrid search for optimal performance.

In [1]:
# make sure .env file is loaded
from dotenv import load_dotenv
print(load_dotenv())

# check whether the OpenAI client works
from openai import OpenAI
openai_client = OpenAI()

True


### Section 1: Embeddings

Turning text into vectors called *embedding* and the obtained vectors are called *embeddings*. The idea comes from [word2vec](https://en.wikipedia.org/wiki/Word2vec). The model learns to place words as points in a multi-dimensional space. Words with similar meanings land close to each other. The same idea works for entire sentences. An embedding model gets text input and returns a fixed-length array of numbers. It is trained so that texts with similar meanings get similar vectors. Here, [sentence-transformers](https://www.sbert.net/) are used, which is a popular open-source library for embeddings. It runs locally without API costs. To install:

```python
uv add sentence-transformers
```

Sentence-transformers supports many models. The right one depends on the task, the language, and the resources. It is better to try a few on models on the data and pick the best working one. Larger models are usually slower, so for a small task, a smalles model should be preferred.

In [2]:
from sentence_transformers import SentenceTransformer
# load a pretrained sentence transformer model
model = SentenceTransformer("all-MiniLM-L6-v2")

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

In [3]:
# sample question embedded
q1 = "Can I still join the course after the start date?"
v1 = model.encode(q1)
print(f"Shape of q1 embedding: {v1.shape}")

# sample document is embedded
d  = "You don't need to register. You're accepted. You can also just start learning and submitting homework without registering."
dv = model.encode(d)
print(f"Shape of d embedding: {dv.shape}")

# compare the sample question against the document with dot product
print(f"Cosine similarity of q1 and the document: {v1.dot(dv)}")

Shape of q1 embedding: (384,)
Shape of d embedding: (384,)
Cosine similarity of q1 and the document: 0.32332396507263184


In [4]:
# a totally unrelated sample question embedded
q2 = "How to install Docker on Windows?"
v2 = model.encode(q2)
print(f"Shape of q2 embedding: {v2.shape}")

# compare the second sample question against the document with dot product
print(f"Cosine similarity of q2 and the document: {v2.dot(dv)}")

Shape of q2 embedding: (384,)
Cosine similarity of q2 and the document: 0.019730573520064354


Two vectors with similar values point to texts about similar things. The score is ear 0 which means the vectors are orthogonal, hence totally unrelated. The initial question has a higher similarity. 

**Cosine similarity:**
The selected all-MiniLM-L6-v2 model outputs normalized vectors. Hence, the dot product equals to the cosine similarity. Cosine similarity measures the angle between two vectors, ignoring their length:

- 1.0 = same direction (similar)
- 0.0 = perpendicular (unrelated)
- -1.0 = opposite direction (opposite meaning)

Formally, if theta is the angle between two vectors, cosine similarity is cos(theta):

- cos(0) = 1 - vectors point in the same direction
- cos(90) = 0 - vectors are perpendicular
- cos(180) = -1 - vectors point in opposite directions

In practice, getting a cosine similarity below 0 is very rare. The embedding model maps text to a region of the vector space where most vectors have positive components. There is no concept of "opposite meaning" that maps to a vector pointing the other way.

### Section 2: Embedding the Dataset

This section loads the same FAQ data from the first module and apply embedding to the whole dataset. Each document in the FAQ dataset is a Python dictionary with a question and an answer. To match a query against the question text and the answer text in the index, both the question and the answer will be embedded together.

In [5]:
from ingest import load_faq_data

# load FAQ data
documents = load_faq_data()

# check a sample
documents[10]

{'course': 'data-engineering-zoomcamp',
 'section': 'General Course-Related Questions',
 'question': 'Course: How many hours per week am I expected to spend on this course?',
 'answer': 'It depends on your background and previous experience with modules. It is expected to require about 5 - 15 hours per week.\n\nYou can also calculate it yourself using [this data](https://github.com/DataTalksClub/zoomcamp-analytics/tree/main/data/de-zoomcamp-2023) and then update this answer.',
 'doc_id': '316180784f'}

In [6]:
texts = []

for doc in documents:
    text = doc["question"] + " " + doc["answer"]
    texts.append(text)

# total number of texts
print(f"Total texts: {len(texts)}")

# check a sample
texts[10]

Total texts: 1350


'Course: How many hours per week am I expected to spend on this course? It depends on your background and previous experience with modules. It is expected to require about 5 - 15 hours per week.\n\nYou can also calculate it yourself using [this data](https://github.com/DataTalksClub/zoomcamp-analytics/tree/main/data/de-zoomcamp-2023) and then update this answer.'

In [7]:
# import tqdm to watch the progress (uv add tqdm)
from tqdm.auto import tqdm

batch_size = 50
vectors = []

# chunk the dataset into batches of 50 and encode each batch
for i in tqdm(range(0, len(texts), batch_size)):
    batch = texts[i:i + batch_size]
    batch_vectors = model.encode(batch)
    vectors.extend(batch_vectors)

# total number of vectors
print(f"Total vectors: {len(vectors)}")

  0%|          | 0/27 [00:00<?, ?it/s]

Total vectors: 1350


In [8]:
import numpy as np

# turn vectors into a matrix where rows are documents (vectors), and cols are dimensions of them
X = np.array(vectors)

print(f"There are {X.shape[0]} vectors with {X.shape[1]} dimensions.")

There are 1350 vectors with 384 dimensions.


### Section 3: Vector Search

The matrix `X` contains all the document embeddings. Rather than checking the cosine similarity of a question with each document with a vector-vector multiplication, it is much simpler and faster to use matrices and perform a matrix-vector multiplication.

In [9]:
# define a sample question and its embedding
query = "Can I still join the course after the start date?"
v_query = model.encode(query)

# get scores of all documents for the current question at hand
scores = X.dot(v_query)

# do not use loops for it, it is much slower
# scores = [v_query.dot(X[i]) for i in range(len(X))]

# get the highest scored document, which is the most similar
idx = np.argmax(scores)
print(f"The most similar document index: {idx}, score: {scores[idx]}")
documents[idx]

The most similar document index: 2, score: 0.7629410028457642


{'course': 'data-engineering-zoomcamp',
 'section': 'General Course-Related Questions',
 'question': 'Course: Can I still join the course after the start date?',
 'answer': "Yes, even if you don't register, you're still eligible to submit the homework.\n\nBe aware, however, that there will be deadlines for turning in homeworks and the final projects. So don't leave everything for the last minute.",
 'doc_id': '3f1424af17'}

The answer to a question can be spread across several documents. One holds part of it, another fills in the rest. Sometimes the top result is not the right one but the second is. Hence all 5 top results are sent to the LLM to be combined. The number can be later arranged by evaluating the search quality.

In [10]:
# return top 5 scores (argsort sorts from lowest to highest)
top5 = np.argsort(scores)[-5:]

# sort top5 scores from highest to lowest
top5 = top5[::-1]

# or as a one line turning min-sort to max-sort: top5 = np.argsort(-scores)[:5]

# show the best 5 results
for idx in top5:
    print(scores[idx])
    print(documents[idx])
    print()

0.762941
{'course': 'data-engineering-zoomcamp', 'section': 'General Course-Related Questions', 'question': 'Course: Can I still join the course after the start date?', 'answer': "Yes, even if you don't register, you're still eligible to submit the homework.\n\nBe aware, however, that there will be deadlines for turning in homeworks and the final projects. So don't leave everything for the last minute.", 'doc_id': '3f1424af17'}

0.7579372
{'course': 'mlops-zoomcamp', 'section': 'General Course-Related Questions', 'question': 'Course - Can I still join the course after the start date?', 'answer': "Yes, even if you don't register, you're still eligible to submit the homeworks as long as the form is still open and accepting submissions.\n\nBe aware, however, that there will be deadlines for turning in the final projects. So don't leave everything to the last minute.", 'doc_id': '2d8b16c2a0'}

0.7192131
{'course': 'machine-learning-zoomcamp', 'section': 'General Course-Related Questions', 

### Section 4: Vector Search with minsearch

This section handles the vector search using the minsearch library. Rather than performing all the tasks of embeddings, computing products, finding and sorting the best matches, an API will be used instead to perform the vector search.

In [11]:
from minsearch import VectorSearch

# index the documents and vectors, define keyword_fields to allow filtering later
vindex = VectorSearch(keyword_fields=["course"])
vindex.fit(X, documents)

In [12]:
# search for a sample question
query = "I just discovered the course. Can I still join it?"
query_vector = model.encode(query)
results = vindex.search(query_vector, num_results=5)

# show the top result
results[0]

{'course': 'llm-zoomcamp',
 'section': 'General Course-Related Questions',
 'question': 'I just discovered the course. Can I still join?',
 'answer': 'Yes, but if you want to receive a certificate, you need to submit your project while we’re still accepting submissions.',
 'doc_id': '74eb249bbf'}

In [13]:
# filter search by keyword_field
results = vindex.search(
    query_vector,
    filter_dict={"course": "llm-zoomcamp"},
    num_results=5
)

# show the top result
results[0]

{'course': 'llm-zoomcamp',
 'section': 'General Course-Related Questions',
 'question': 'I just discovered the course. Can I still join?',
 'answer': 'Yes, but if you want to receive a certificate, you need to submit your project while we’re still accepting submissions.',
 'doc_id': '74eb249bbf'}

### Section 5: RAG with Vector Search

The only difference to RAG implementation in module 1 is how the search is performed. Previously keyword search was used, which will be replaced with vector search here. Below is how module 1 handled RAG assistants with keyword search.

In [14]:
from ingest import load_faq_data, build_index
from rag_helper import RAGBase

# download and index the data
documents = load_faq_data()
index = build_index(documents)

# create a RAG assistant which works with keyword search
assistant = RAGBase(
    index=index,
    llm_client=openai_client,
)

# try keyword search
query = "I just found out about the program, can I still sign up?"
assistant.rag(query)

'Yes, you can still join. If you want to receive a certificate, you need to submit your project while submissions are still being accepted.'

To replace the keyword search with vector search, subclass the existing class and redefine the search as a vector search.

In [15]:
# RAG with vector search
class RAGVector(RAGBase):

    def __init__(self, embedder, **kwargs):
        super().__init__(**kwargs)
        # extra argument for sentence trasformer
        self.embedder = embedder

    def search(self, query, num_results=5):
        query_vector = self.embedder.encode(query)
        filter_dict = {"course": self.course}

        return self.index.search(
            query_vector,
            num_results=num_results,
            filter_dict=filter_dict
        )


# initialize the RAG assistant with vector search
vector_assistant = RAGVector(
    embedder=model,
    index=vindex,
    llm_client=openai_client,
)

# try vector search
query = "the program has already begun, can I still sign up?"
vector_assistant.rag(query)

'Yes, you can still join. The course can be started even after it has begun, and you can begin learning and submitting while the submission form is open. If you want a certificate, make sure to submit your project before submissions close.'

### Section 6: Vector Search with sqlitesearch

While using minsearch for exact nearest neighbor (NN) vector search works on small datasets, it suffers from critical scalability issues. First, indexing is slow because it runs a neural network over every document, and rebuilding this index on every startup becomes inefficient. Second, keeping all vector embeddings in memory does not scale as datasets grow larger. Finally, its brute-force search method, which compares a query against every single document vector, becomes too slow past roughly 10,000 documents. To address these limitations, transitioning to approximate nearest neighbor (ANN) search is necessary because it improves speed by narrowing the search to a region of likely matches rather than scanning the entire dataset.

- **NN (exact)**:    compare query against ALL documents -> top 5
- **ANN (approx)**:  narrow down to a region -> compare within region -> top 5

`sqlitesearch` is the persistent sibling of `minsearch`, and it solves both problems at once. It does vector search through its `VectorSearchIndex` class. It stores vectors in `SQLite`, a real on-disk database, and uses ANN strategies for retrieval. Because the data lives on disk, one process can write the vectors and another can read them back. It can be added with:

```python
uv add sqlitesearch
```

In [16]:
from sqlitesearch import VectorSearchIndex

# create the index with sqlite search
vs_index = VectorSearchIndex(
    keyword_fields=["course"],
    mode="ivf",
    db_path="faq_vectors2.db"
)

# fit the index with vectors and documents
vs_index.fit(vectors, documents)

`sqlitesearch` supports three ANN modes:

- `lsh` (default): up to 100K vectors, random hyperplane projections
- `ivf`: 10K-500K vectors, K-means clustering
- `hnsw`: 10K-1M+ vectors, proximity graph (highest recall)
For small datasets, `lsh` is fine. All modes use two-phase search: approximate candidate retrieval, then exact cosine similarity reranking.

In [18]:
query = "I just discovered the course. Can I still join it?"
# encode the query
query_vector = model.encode(query)

# perform search with sqlitesearch
results = vs_index.search(query_vector, num_results=5)

# show the output
results

[{'course': 'llm-zoomcamp',
  'section': 'General Course-Related Questions',
  'question': 'I just discovered the course. Can I still join?',
  'answer': 'Yes, but if you want to receive a certificate, you need to submit your project while we’re still accepting submissions.',
  'doc_id': '74eb249bbf'},
 {'course': 'machine-learning-zoomcamp',
  'section': 'General Course-Related Questions',
  'question': 'The course has already started. Can I still join it?',
  'answer': 'Yes, you can. Even though you missed the start date, you can register for the course. You won’t be able to submit some of the homeworks, but you can still take part in the course.\n\nIn order to get a certificate, you need to submit 2 out of 3 course projects and review 3 peers by the deadline. It means that if you join the course at the end of November and manage to work on two projects, you will still be eligible for a certificate.',
  'doc_id': '41aabbd7c5'},
 {'course': 'mlops-zoomcamp',
  'section': 'General Cour

In [19]:
# filter by course
results = vs_index.search(
    query_vector,
    filter_dict={"course": "llm-zoomcamp"},
    num_results=5
)

# show the output
results

[{'course': 'llm-zoomcamp',
  'section': 'General Course-Related Questions',
  'question': 'I just discovered the course. Can I still join?',
  'answer': 'Yes, but if you want to receive a certificate, you need to submit your project while we’re still accepting submissions.',
  'doc_id': '74eb249bbf'},
 {'course': 'llm-zoomcamp',
  'section': 'General Course-Related Questions',
  'question': 'Certificate: Can I follow the course in a self-paced mode and get a certificate?',
  'answer': 'No, you can only get a certificate if you finish the course with a "live" cohort.\n\nWe don\'t award certificates for the self-paced mode. The reason is you need to peer-review 3 capstone(s) after submitting your project.\n\nYou can only peer-review projects at the time the course is running; after the form is closed and the peer-review list is compiled.',
  'doc_id': '69d122f12e'},
 {'course': 'llm-zoomcamp',
  'section': 'General Course-Related Questions',
  'question': 'When will the course be offere

In [23]:
# close the index when no more search is needed
vs_index.close()

In [ ]:
# reopen the index without re-computing embeddings (would also work on a new Python session)
from sentence_transformers import SentenceTransformer
from sqlitesearch import VectorSearchIndex

model = SentenceTransformer("all-MiniLM-L6-v2")

vs_index = VectorSearchIndex(
    keyword_fields=["course"],
    mode="ivf",
    db_path="faq_vectors2.db"
)

# no fit is needed because index is already built and waiting on the disk
query_vector = model.encode("How do I run Kafka?")
results = vs_index.search(query_vector, num_results=5)

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

In [ ]:
results

[{'course': 'data-engineering-zoomcamp',
  'section': 'Module 7: Streaming',
  'question': 'Java Kafka: How to run producer/consumer/kstreams/etc in terminal',
  'answer': 'In the project directory, run:\n\n```bash\njava -cp build/libs/<jar_name>-1.0-SNAPSHOT.jar:out src/main/java/org/example/JsonProducer.java\n```',
  'doc_id': '5ca6890c1a'},
 {'course': 'data-engineering-zoomcamp',
  'section': 'Module 7: Streaming',
  'question': 'Java Kafka: When running the producer/consumer/etc java scripts, no results retrieved or no message sent',
  'answer': 'For example, when running `JsonConsumer.java`, you might see:\n\n```\nConsuming form kafka started\n\nRESULTS:::0\n\nRESULTS:::0\n\nRESULTS:::0\n```\n\nOr when running `JsonProducer.java`, you might encounter:\n\n```\nException in thread "main" java.util.concurrent.ExecutionException: org.apache.kafka.common.errors.SaslAuthenticationException: Authentication failed\n```\n\n**Solution:**\n\n1. Ensure the `StreamsConfig.BOOTSTRAP_SERVERS_CO

- `minsearch` `VectorSearch`: in-memory (numpy), exact cosine similarity, must re-compute embeddings on startup, good for experiments and notebooks
- `sqlitesearch` `VectorSearchIndex`: persistent (`SQLite` .db file), ANN (LSH/IVF/HNSW) with exact rerank, can open an existing index, good for projects and persistence

`sqlitesearch` is a simple library whose only dependencies are SQLite and numpy. It runs on any host that offers a free SQLite database, where a dedicated vector database would cost extra. However it is for course work and not for production level implementations.

### Section 7: Vector Search with PGVector

Many production databases, including Elasticsearch, Qdrant, and Chroma, support vector search, but this approach utilizes Postgres due to its existing familiarity in workplace and course settings. The foundational concept remains identical to sqlitesearch, changing only the underlying database technology. To enable vector similarity search, Postgres requires the installation of the pgvector extension. By utilizing this setup, users gain access to standard production-grade features such as concurrent access, transactions, and support for large datasets. Ultimately, this Postgres configuration with pgvector will be deployed and run within a Docker container.

1. Start Postgres with pgvector by pulling the image and starting a container:
```python
docker run -it \
    --name pgvector \
    -e POSTGRES_USER=user \
    -e POSTGRES_PASSWORD=pswd \
    -e POSTGRES_DB=faq \
    -v pgvector_data:/var/lib/postgresql/data \
    -p 5432:5432 \
    pgvector/pgvector:pg17
```

2. Install the Python client by installing the driver:
```python
uv add psycopg[binary]
```
psycopg (v3) will be used to connect and run queries. 
Note: v3 is different from psycopg2. psycopg v3 supports conn.execute() directly without creating a cursor.

The implementation for this section is in the *pgvector.ipynb file

Here's how PGVector compares with the two tools used earlier:

- minsearch: no setup, in-memory, best for notebooks and experiments
- sqlitesearch: no setup, SQLite file persistence, best for pet projects
- PGVector: requires Docker, Postgres database with concurrent access, handles millions of records, best for production systems

Reach for PGVector when you need production features:
- concurrent reads and writes
- transactions
- integration with an existing Postgres-based application

### Section 8: Using ONNX Runtime instead of PyTorch

Moving to production requires minimizing deployment size and dependency overhead by replacing heavy frameworks. While sentence-transformers is suitable for development, it introduces a 4.8 GB virtual environment with 58 packages due to its reliance on PyTorch and Nvidia libraries. In contrast, ONNX Runtime serves the exact same models and embeddings while requiring only 147 MB and 27 packages. This transition results in a deployment that is 33 times smaller without sacrificing accuracy. Because pre-converted ONNX versions of popular models are typically available for download, developers rarely need to perform the conversion themselves. Create a separate project for this part:

```python
mkdir llm__zoomcamp_onnx && cd llm_zoomcamp_onnx
uv init --no-workspace
uv add onnxruntime tokenizers numpy tqdm minsearch
uv add --dev huggingface-hub jupyter
```

`huggingface-hub` is only needed to download the model. At runtime we'll need `onnxruntime`, `tokenizers`, and `numpy`. Register a kernel for this project.

```python
uv run python -m ipykernel install --user --name llm_zoomcamp_onnx --display-name "llm_zoomcamp_onnx"
```

Use the `download.py` script from the `embed/` directory to fetch the ONNX model from HuggingFace.
```python
uv run python download.py
```

Use the `embedder.py` script from the `embed/` directory for generating embeddings. The provided code file can be integrated directly into a project to replace the heavy PyTorch dependency while maintaining the identical encode interface. Internally, the script executes four specific operations to generate embeddings. First, it tokenizes the input text, converting it into integer IDs and attention masks. Next, it executes the ONNX model graph directly on the CPU to extract token-level embeddings. Finally, it performs mean pooling to average these embeddings based on the attention mask, followed by L2 normalization to enable vector comparison via a simple dot product.

### Section 9: Next Steps

This module focused on understanding and generating embeddings, building vector search with tools like NumPy, PostgreSQL, and pgvector, and integrating them into a RAG pipeline. Although popular tutorials often promote vector search initially, it introduces significant compute, storage, and dependency overhead that may not be necessary on day one. A strategic progression involves starting with simple text search (v1), adding vector search only when users query using different vocabulary than the documents (v2), and eventually merging them into a hybrid search system (v3). Moving between these phases should always be justified by objective evaluation metrics rather than marginal gains. Advanced retrieval quality can be achieved later through hybrid combinations like Reciprocal Rank Fusion (RRF) and secondary reranking models.